# Nested Clustered Optimisation (NCO) Allocator

The allocator will do the following:
- Denoise and detone the correlation matrix
- Cluster ETFs through the ONC method on the correlation from previous step
- Denoise the covariance matrix
- Run the min-variance optimization within cluster's own covariance 
- Run the min-variance optimization accross clusters as if they were a single assets; their covariance is calculated from intra-cluster weights
- The final weights will be a product of the two weights, between clusters and withing clusters

Note; to perform the steps, the min_var fucntion in allocators had to be restructured to allow for flexibility in the use of a simple covariance or a denoised covariance

In [31]:
# Imports
import pandas as pd
import numpy as np
from vportfolio.params import CAP, LOOKBACK
from vportfolio import load_data, engine, denoising, clustering, allocators

In [32]:
# a function to do all the math related to the NCO allocator
# will return an optimized set of final weights

def nco_weights(cov, labels):
    clusters = labels.unique()
    W = pd.DataFrame(0.0, index=cov.index, columns=clusters)
    # Split the matrix in sub groups based on NCO's labels
    for g in clusters:
        mask = cov.index.isin(labels.index[labels == g]) # I use a mask to filter the entire matrix into sub groups
        sub_cov = cov.loc[mask, mask]
        # Run a min_var allocator to obtain sub groups composition
        W.loc[mask, g] = allocators.min_var_weights(sub_cov,cap=None) # cap set to none or the function will fail as there need to be at least 5 assets
    # Calculate the reduced covariance matrix between clusters
    cov_r = W.T @ cov @ W
    # Run min_var allocator between clusters
    between_w = allocators.min_var_weights(cov_r, cap=None)
    # final asset weights
    return W @ between_w

In [33]:
# testing function

assets = ["A", "B", "C", "D"] # 4 fake assets
cov_test = pd.DataFrame(np.diag([1, 4, 1, 4]), index=assets, columns=assets) # fake cov matrix
labels_test = pd.Series([1, 1, 0, 0], index=assets) # 2 clusters
w = nco_weights(cov_test, labels_test)

# 1. always true: weights sum to 1, all positive
assert np.isclose(w.sum(), 1)
assert (w > 0).all()
assert list(w.index) == assets
# 2. known answer: 
# within cluster weights = 1/var, so A = 1 and B = 0.25. Normalized it becomes 0.8 and 0.2 (same for C and D)
# identical clusters get 0.5 each
inv_var = 1 / np.diag(cov_test)
expected = inv_var / inv_var.sum()
assert np.allclose(w, expected)
# 3. inputs matter: raise A's variance to 2
cov_test2 = cov_test.copy()
cov_test2.loc["A", "A"] = 2
w2 = nco_weights(cov_test2, labels_test)
# diagonal cov -> NCO collapses to inverse variance
inv_var = 1 / np.diag(cov_test2)
expected = inv_var / inv_var.sum()
assert np.allclose(w2, expected)
assert w2["A"] < w["A"]   # riskier A gets less weight

print("nco_weights passed all tests")

nco_weights passed all tests


In [34]:
print(w.values)
print(expected)
print(np.abs(w.values - expected).max())

[0.4 0.1 0.4 0.1]
[0.25  0.125 0.5   0.125]
0.15000000223517412


In [35]:
sub = cov_test.loc[["A", "B"], ["A", "B"]]
print(allocators.min_var_weights(sub, cap=None))   # should be 0.8 / 0.2
print(allocators.min_var_weights(sub, cap=1.0))

A    0.8
B    0.2
dtype: float64
A    0.8
B    0.2
dtype: float64


In [36]:
print(allocators.min_var_weights(sub * 1e-4, cap=None))   # expect 0.8 / 0.2

A    0.8
B    0.2
dtype: float64


In [37]:
# helper function to apply the cap to the NCO allocator
# returns a series of capped weights
def apply_cap(w, cap=CAP):
    w = w.copy()
    for _ in range(100):
        over = w > cap
        if not over.any():
            break
        excess = (w[over] - cap).sum()
        w[over] = cap
        w[~over] += excess * w[~over] / w[~over].sum() # pro|-rata to uncapped names
    return w

In [38]:
# function testing
w = pd.Series([0.5, 0.2, 0.1, 0.1, 0.1])
cap = 0.3
capped = apply_cap(w, cap)
over = w > cap
# capped names sit exactly at the cap
assert np.allclose(capped[over], cap)
# uncapped names keep their relative proportions
assert np.allclose(capped[~over] / capped[~over].sum(),
                   w[~over] / w[~over].sum())
# total still 1, nothing over the cap
assert np.isclose(capped.sum(), 1)
assert (capped <= cap + 1e-12).all()

print("apply_cap passed all tests")

apply_cap passed all tests


In [39]:
# Obtain the final nco weights after adjusting the inputs
# will return a series with capped weights
def nco(window, cap = CAP):
    t = window.shape[0]
    corr_dt = denoising.detone_corr(denoising.denoise_corr(window.corr(), t)) # correlation: denoise → detone
    labels = clustering.onc(corr_dt) # ONC on that correlation
    cov = denoising.denoised_cov(window) # covariance: denoised only
    w = nco_weights(cov, labels).reindex(window.columns) # optimise
    if len(w) * cap < 1:
      raise ValueError(f"cap {cap} infeasible for {len(w)} assets")
    return apply_cap(w, cap)

In [40]:
# Function testing

# build 3 identical blocks of 4 assets: shared factor + own noise
rng = np.random.default_rng(0)
n_days, blocks, per_block = 500, 3, 4
factors = rng.normal(0, 0.01, (n_days, blocks))
returns = np.hstack([factors[:, [b]] + rng.normal(0, 0.005, (n_days, per_block))
                     for b in range(blocks)])
n_assets = blocks * per_block
window = pd.DataFrame(returns, columns=[f"X{i}" for i in range(n_assets)])

w = nco(window, cap=0.2)

# 1. always true: same order, no gaps, sums to 1, within bounds
assert list(w.index) == list(window.columns)
assert not w.isna().any()
assert np.isclose(w.sum(), 1)
assert (w >= 0).all() and (w <= 0.2 + 1e-12).all()

# 2. known answer
true_labels = pd.Series(np.repeat(range(blocks), per_block), index=window.columns)
t = window.shape[0]
labels = clustering.onc(denoising.detone_corr(denoising.denoise_corr(window.corr(), t)))

# 2a. ONC recovers the blocks (label numbers may differ, groupings must match)
assert pd.crosstab(labels, true_labels).gt(0).sum(axis=1).eq(1).all()
assert labels.nunique() == blocks

# 2b. each block gets ~1/3 (tight: noise averages out within a block)
assert np.allclose(w.groupby(true_labels).sum(), 1 / blocks, atol=0.03)

# 2c. each asset near 1/12 (loose: min-var amplifies noise in correlated assets)
assert np.allclose(w, 1 / n_assets, atol=0.05)

# 3. scale invariance: doubling returns leaves corr unchanged, cov x4 -> same weights
assert np.allclose(nco(window * 2, cap=0.2), w, atol=1e-6)

# 4. deterministic: same input, same output. in case NCO uses k-means with a random seed
assert np.allclose(nco(window, cap=0.2), w)

print("nco passed all tests")

nco passed all tests


All functions are working so as usual we run our sanity check on a fixed window

In [41]:
# let's import the data
data = load_data.load_universe()
# let's take the last date of the research period
decision_date = data["rebal_dates"].iloc[-1]
# calculate the lookback window
window = engine.lookback_window(data["returns"], data["eligible"], decision_date, n=LOOKBACK)
w_nco = nco(window)
w_mv = allocators.min_var(window, denoise=True)
pd.DataFrame({"nco": w_nco, "min_var": w_mv}).round(3)

,nco,min_var
Ticker,,
SPY,0.00,0.018
QQQ,0.00,0.003
IWM,0.00,0.000
EFA,0.00,0.003
EEM,0.00,0.000
TLT,0.00,0.000
IEF,0.00,0.190
SHY,0.20,0.200
TIP,0.00,0.015


Strange results considering that EM Local, one of the most volatile in teh list, is at 20%

let's take a look at uncapped weights

In [42]:
t = window.shape[0]
labels = clustering.onc(denoising.detone_corr(denoising.denoise_corr(window.corr(), t)))

w_raw = nco_weights(denoising.denoised_cov(window), labels).reindex(window.columns)
pd.DataFrame({"raw": w_raw, "capped": w_nco, "cluster": labels}).round(3).sort_values("raw", ascending=False)

,raw,capped,cluster
Ticker,,,
SHY,0.916,0.20,1
UUP,0.055,0.20,0
HYG,0.019,0.20,0
EMLC,0.005,0.20,1
FXA,0.003,0.15,1
DBC,0.001,0.05,0
EEM,0.000,0.00,0
IWM,0.000,0.00,0
QQQ,0.000,0.00,0


Yes, the cap is creating the weights artificially. NCO puts 91.6% in SHY. I will adjust the nco formula to decide where the excess allocation goes excluding the fixed name.

In other words, if SHY is capped and out of the allocation, where would the remaining 80% go?

In [43]:
# a function to run NCO repeatedly, locking any name that goes above cap at the cap and removing it from the problem
# returns a series of weights indexed by assets
def nco_capped(cov, labels, cap=CAP):
    if len(cov) * cap < 1: # feasibility check
        raise ValueError(f"cap {cap} infeasible for {len(cov)} assets")
    w = pd.Series(0.0, index=cov.index)
    free = cov.index
    budget = 1.0
    while True:
        sub = nco_weights(cov.loc[free, free], labels[free]) * budget # run NCO on free assets only, and use budget to scale them down to what is actually left
        over = sub > cap + 1e-12 # boolean mask of which assets went over the cap (excluding rounding noise)
        if not over.any():
            w[free] = sub
            return w
        w[sub.index[over]] = cap
        budget -= cap * over.sum()
        free = sub.index[~over]

In [44]:
# testing function

assets = ["A", "B", "C", "D", "E"]
variances = np.array([1, 2, 8, 8, 8])
cov_cap = pd.DataFrame(np.diag(variances), index=assets, columns=assets)
labels_cap = pd.Series([0, 0, 1, 1, 1], index=assets)
cap = 0.3

w = nco_capped(cov_cap, labels_cap, cap)

# 1. always true
assert np.isclose(w.sum(), 1)
assert (w >= 0).all() and (w <= cap + 1e-12).all()
# 2. known answer: locked names at the cap, free names ∝ 1/variance
locked = np.isclose(w, cap)
assert list(w.index[locked]) == ["A", "B"]          # two rounds of locking
inv_var = pd.Series(1 / variances, index=assets)[~locked]
assert np.allclose(w[~locked], inv_var / inv_var.sum() * (1 - cap * locked.sum()))
# 3. cap not binding -> identical to plain nco_weights
assert np.allclose(nco_capped(cov_cap, labels_cap, cap=1.0), nco_weights(cov_cap, labels_cap))
# 4. infeasible cap raises
try:
    nco_capped(cov_cap, labels_cap, cap=0.1)
    raise AssertionError("should have raised")
except ValueError:
    pass

print("nco_capped passed all tests")

nco_capped passed all tests


In [45]:
# Obtain the final nco weights after adjusting the inputs
# will return a series with capped weights
def nco(window, cap=CAP):
    t = window.shape[0]
    corr_dt = denoising.detone_corr(denoising.denoise_corr(window.corr(), t)) # correlation: denoise → detone
    labels = clustering.onc(corr_dt) # ONC on that correlation
    cov = denoising.denoised_cov(window) # covariance: denoised only
    return nco_capped(cov, labels, cap).reindex(window.columns) # NCO with the cap applied inside

In [46]:
# Function testing

# build 3 identical blocks of 4 assets: shared factor + own noise
rng = np.random.default_rng(0)
n_days, blocks, per_block = 500, 3, 4
factors = rng.normal(0, 0.01, (n_days, blocks))
returns = np.hstack([factors[:, [b]] + rng.normal(0, 0.005, (n_days, per_block))
                     for b in range(blocks)])
n_assets = blocks * per_block
window = pd.DataFrame(returns, columns=[f"X{i}" for i in range(n_assets)])

w = nco(window, cap=0.2)

# 1. always true: same order, no gaps, sums to 1, within bounds
assert list(w.index) == list(window.columns)
assert not w.isna().any()
assert np.isclose(w.sum(), 1)
assert (w >= 0).all() and (w <= 0.2 + 1e-12).all()

# 2. known answer
true_labels = pd.Series(np.repeat(range(blocks), per_block), index=window.columns)
t = window.shape[0]
labels = clustering.onc(denoising.detone_corr(denoising.denoise_corr(window.corr(), t)))

# 2a. ONC recovers the blocks (label numbers may differ, groupings must match)
assert pd.crosstab(labels, true_labels).gt(0).sum(axis=1).eq(1).all()
assert labels.nunique() == blocks

# 2b. each block gets ~1/3 (tight: noise averages out within a block)
assert np.allclose(w.groupby(true_labels).sum(), 1 / blocks, atol=0.03)

# 2c. each asset near 1/12 (loose: min-var amplifies noise in correlated assets)
assert np.allclose(w, 1 / n_assets, atol=0.05)

# 3. scale invariance: doubling returns leaves corr unchanged, cov x4 -> same weights
assert np.allclose(nco(window * 2, cap=0.2), w, atol=1e-6)

# 4. deterministic: same input, same output. in case NCO uses k-means with a random seed
assert np.allclose(nco(window, cap=0.2), w)

print("nco passed all tests")

nco passed all tests


In [48]:
# let's import the data
data = load_data.load_universe()
# let's take the last date of the research period
decision_date = data["rebal_dates"].iloc[-1]
# calculate the lookback window
window = engine.lookback_window(data["returns"], data["eligible"], decision_date, n=LOOKBACK)

t = window.shape[0]
labels = clustering.onc(denoising.detone_corr(denoising.denoise_corr(window.corr(), t)))
cov = denoising.denoised_cov(window)

comparison = pd.DataFrame({
    "nco_raw": nco_weights(cov, labels).reindex(window.columns),
    "nco": nco(window),
    "min_var": allocators.min_var(window, denoise=True),
    "cluster": labels,
}).round(3)
comparison.sort_values("nco", ascending=False)

,nco_raw,nco,min_var,cluster
Ticker,,,,
IEF,0.000,0.200,0.190,1
SHY,0.916,0.200,0.200,1
UUP,0.055,0.200,0.200,0
TIP,0.000,0.145,0.015,1
FXF,0.000,0.115,0.172,1
FXY,0.000,0.069,0.132,1
HYG,0.019,0.040,0.052,0
EMB,0.000,0.012,0.000,1
EMLC,0.005,0.011,0.000,1


The result looks more meaningful
- The cap artifact is gone
- The excess went were it should (after SHY: IEF, UUP, TIP, etc)
- NCO and min-var now seem more alligned; the main difference is that is that NCO perfers TIP vs yen and franc. That is because NCO allocates within the defensive cluster as one group